# Notebook 03 — Feature Engineering

Builds a match-level feature table for model training. For every match we compute — **just before kick-off** — features for both the home and away team:

| Feature | Description |
|---|---|
| `elo` | ELO rating (K=40 WC/continental, K=30 qualifier, K=20 friendly) |
| `elo_diff` | home ELO − away ELO |
| `form_win_rate` | win rate over last 10 matches (all competitions) |
| `form_goals_scored` | avg goals scored last 10 matches |
| `form_goals_conceded` | avg goals conceded last 10 matches |
| `tournament_weight` | match importance (1–5) |
| `is_neutral` | 0 = home ground, 1 = neutral venue |
| `wc_titles` | historical World Cup titles (static) |
| `wc_appearances` | historical WC tournament appearances (static) |

Output: `data/processed/matches_features.csv` + `data/processed/elo_snapshot.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

PROC = Path('data/processed')

df = pd.read_csv(PROC / 'results_clean.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f'Loaded {len(df):,} matches')
print(f'Columns: {list(df.columns)}')

Loaded 49,373 matches
Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'year', 'month', 'outcome', 'goal_diff', 'total_goals', 'tournament_weight', 'is_competitive']


## 1. ELO Rating System

Starting ELO = 1000 for all teams. K-factor scales with match importance.

In [2]:
def k_factor(weight: int) -> int:
    """K-factor by tournament importance weight."""
    return {5: 40, 4: 35, 3: 30, 2: 25, 1: 20}.get(weight, 20)

def expected_score(elo_a: float, elo_b: float) -> float:
    return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

# Initialise
elo: dict[str, float] = defaultdict(lambda: 1000.0)

home_elo_before, away_elo_before = [], []

for _, row in df.iterrows():
    ht, at = row['home_team'], row['away_team']
    h_elo, a_elo = elo[ht], elo[at]

    home_elo_before.append(h_elo)
    away_elo_before.append(a_elo)

    # Actual scores (1=win, 0.5=draw, 0=loss) for home side
    actual = {0: 1.0, 1: 0.5, 2: 0.0}[row['outcome']]
    exp = expected_score(h_elo, a_elo)
    K = k_factor(row['tournament_weight'])

    # Margin-of-victory multiplier (capped)
    goal_diff = abs(row['goal_diff'])
    mov = np.log(goal_diff + 1) + 1  # log multiplier

    delta = K * mov * (actual - exp)
    elo[ht] += delta
    elo[at] -= delta

df['home_elo'] = home_elo_before
df['away_elo'] = away_elo_before
df['elo_diff'] = df['home_elo'] - df['away_elo']

print('ELO computed for all matches.')
print(f'\nTop 10 current ELO ratings:')
top_elo = sorted(elo.items(), key=lambda x: x[1], reverse=True)[:10]
for team, rating in top_elo:
    print(f'  {team:<25s}  {rating:>7.1f}')

ELO computed for all matches.

Top 10 current ELO ratings:
  Spain                       1733.5
  Argentina                   1687.7
  France                      1669.0
  England                     1606.0
  Brazil                      1587.4
  Germany                     1582.0
  Portugal                    1576.8
  Colombia                    1565.7
  Netherlands                 1558.2
  Morocco                     1548.4


## 2. Rolling Form Features

For each team, compute rolling stats over their last 10 matches before each game.

In [3]:
WINDOW = 10

# Build a per-team match history (all matches, chronological)
# Each entry: (date, win, goals_scored, goals_conceded)
team_history: dict[str, list] = defaultdict(list)

# We need home and away form computed at match time — iterate once to build histories,
# then iterate again to look up the rolling window.
# Since df is sorted by date, we process in order and look back.

# First pass: record results per team
for _, row in df.iterrows():
    ht, at = row['home_team'], row['away_team']
    outcome = row['outcome']
    hs, as_ = row['home_score'], row['away_score']

    # home team entry
    team_history[ht].append({
        'date': row['date'],
        'win':  int(outcome == 0),
        'draw': int(outcome == 1),
        'goals_scored':   hs,
        'goals_conceded': as_,
    })
    # away team entry
    team_history[at].append({
        'date': row['date'],
        'win':  int(outcome == 2),
        'draw': int(outcome == 1),
        'goals_scored':   as_,
        'goals_conceded': hs,
    })

print(f'Built history for {len(team_history)} teams.')

Built history for 335 teams.


In [4]:
# Convert histories to DataFrames for fast rolling lookup
team_df: dict[str, pd.DataFrame] = {}
for team, records in team_history.items():
    tdf = pd.DataFrame(records).sort_values('date').reset_index(drop=True)
    team_df[team] = tdf

def rolling_form(team: str, before_date: pd.Timestamp, window: int = WINDOW) -> dict:
    """Return rolling stats over last `window` matches before `before_date`."""
    tdf = team_df.get(team)
    if tdf is None:
        return {'win_rate': 0.5, 'goals_scored': 1.5, 'goals_conceded': 1.5, 'n_games': 0}
    past = tdf[tdf['date'] < before_date].tail(window)
    n = len(past)
    if n == 0:
        return {'win_rate': 0.5, 'goals_scored': 1.5, 'goals_conceded': 1.5, 'n_games': 0}
    return {
        'win_rate':        past['win'].mean(),
        'goals_scored':    past['goals_scored'].mean(),
        'goals_conceded':  past['goals_conceded'].mean(),
        'n_games':         n,
    }

# Build rolling features for every match (only from 1990 onward for training relevance)
print('Computing rolling form features ...')
train_df = df[df['date'].dt.year >= 1990].copy().reset_index(drop=True)

home_form_rows, away_form_rows = [], []

for _, row in train_df.iterrows():
    home_form_rows.append(rolling_form(row['home_team'], row['date']))
    away_form_rows.append(rolling_form(row['away_team'], row['date']))

home_form_df = pd.DataFrame(home_form_rows).add_prefix('home_form_')
away_form_df = pd.DataFrame(away_form_rows).add_prefix('away_form_')

train_df = pd.concat([train_df.reset_index(drop=True),
                      home_form_df, away_form_df], axis=1)

print(f'Done. Training set: {len(train_df):,} matches')

Computing rolling form features ...


Done. Training set: 32,255 matches


## 3. Static Historical Features

In [5]:
WC_TITLES = {
    'Brazil': 5, 'Germany': 4, 'Italy': 4, 'Argentina': 3,
    'France': 2, 'Uruguay': 2, 'England': 1, 'Spain': 1,
}

WC_APPEARANCES = {
    'Brazil': 22, 'Germany': 20, 'Italy': 18, 'Argentina': 18,
    'Mexico': 17, 'France': 16, 'England': 16, 'Spain': 16,
    'Belgium': 15, 'Uruguay': 14, 'Netherlands': 11, 'Sweden': 12,
    'Japan': 8, 'South Korea': 11, 'United States': 11, 'Portugal': 9,
    'Colombia': 7, 'Australia': 6, 'Croatia': 7, 'Denmark': 6,
    'Switzerland': 12, 'Poland': 9, 'Cameroon': 8, 'Morocco': 6,
    'Nigeria': 7, 'Senegal': 4, 'Iran': 6, 'Saudi Arabia': 6,
    'Ecuador': 4, 'Paraguay': 9, 'Chile': 9, 'Peru': 5,
    'Serbia': 4, 'Austria': 7, 'Scotland': 8, 'Canada': 2,
}

for prefix, col_ht, col_at in [
    ('home', 'home_team', 'wc_titles_home'),
    ('away', 'away_team', 'wc_titles_away'),
]:
    pass  # replaced below

train_df['home_wc_titles']  = train_df['home_team'].map(WC_TITLES).fillna(0).astype(int)
train_df['away_wc_titles']  = train_df['away_team'].map(WC_TITLES).fillna(0).astype(int)
train_df['home_wc_apps']    = train_df['home_team'].map(WC_APPEARANCES).fillna(0).astype(int)
train_df['away_wc_apps']    = train_df['away_team'].map(WC_APPEARANCES).fillna(0).astype(int)

print('Static features added.')
print(train_df[['home_team','home_wc_titles','home_wc_apps']].drop_duplicates().sort_values('home_wc_titles', ascending=False).head(8).to_string(index=False))

Static features added.
home_team  home_wc_titles  home_wc_apps
   Brazil               5            22
  Germany               4            20
    Italy               4            18
Argentina               3            18
  Uruguay               2            14
   France               2            16
  England               1            16
    Spain               1            16


## 4. Final Feature Table

In [6]:
FEATURE_COLS = [
    # ELO
    'home_elo', 'away_elo', 'elo_diff',
    # Rolling form — home
    'home_form_win_rate', 'home_form_goals_scored', 'home_form_goals_conceded',
    # Rolling form — away
    'away_form_win_rate', 'away_form_goals_scored', 'away_form_goals_conceded',
    # Match context
    'tournament_weight', 'neutral',
    # Historical pedigree
    'home_wc_titles', 'away_wc_titles', 'home_wc_apps', 'away_wc_apps',
]
TARGET_COL = 'outcome'

print(f'Training set size before cold-start filter: {len(train_df):,}')
print(f'Columns available: {[c for c in train_df.columns if "form" in c]}')

Training set size before cold-start filter: 32,255
Columns available: ['home_form_win_rate', 'home_form_goals_scored', 'home_form_goals_conceded', 'home_form_n_games', 'away_form_win_rate', 'away_form_goals_scored', 'away_form_goals_conceded', 'away_form_n_games']


In [7]:
# Add cold-start filter using the n_games columns from train_df
features_df = train_df[['date', 'home_team', 'away_team', 'tournament'] +
                        FEATURE_COLS +
                        ['home_form_n_games', 'away_form_n_games', TARGET_COL]].copy()

features_df = features_df[
    (features_df['home_form_n_games'] >= 3) &
    (features_df['away_form_n_games'] >= 3)
].drop(columns=['home_form_n_games', 'away_form_n_games'])

features_df = features_df.dropna(subset=FEATURE_COLS).reset_index(drop=True)

print(f'Final feature table: {len(features_df):,} rows x {len(features_df.columns)} columns')
features_df.tail(4)

Final feature table: 31,974 rows x 20 columns


,date,home_team,away_team,tournament,home_elo,away_elo,elo_diff,home_form_win_rate,home_form_goals_scored,home_form_goals_conceded,away_form_win_rate,away_form_goals_scored,away_form_goals_conceded,tournament_weight,neutral,home_wc_titles,away_wc_titles,home_wc_apps,away_wc_apps,outcome
31970,2026-06-07,Ecuador,Guatemala,Friendly,1541.038396,1118.784466,422.253929,0.3,0.9,0.5,0.2,1.1,2.2,1,1,0,0,4,0,0
31971,2026-06-07,Croatia,Slovenia,Friendly,1513.977827,1269.045357,244.932469,0.7,2.2,1.0,0.2,0.9,1.3,1,0,0,0,7,0,0
31972,2026-06-07,Greece,Italy,Friendly,1344.268490,1477.960143,-133.691653,0.3,1.6,1.5,0.8,2.5,1.0,1,0,0,4,0,18,2
31973,2026-06-07,Denmark,Ukraine,Friendly,1459.412577,1360.978526,98.434051,0.5,2.7,0.9,0.6,1.6,1.5,1,0,0,0,6,0,0


## 5. ELO Snapshot for WC 2026 Teams

In [8]:
WC2026_TEAMS = [
    'United States', 'Mexico', 'Canada', 'Panama', 'Honduras', 'Jamaica',
    'Argentina', 'Brazil', 'Colombia', 'Ecuador', 'Uruguay', 'Venezuela',
    'Germany', 'France', 'Spain', 'England', 'Portugal', 'Netherlands',
    'Belgium', 'Italy', 'Croatia', 'Austria', 'Switzerland', 'Denmark',
    'Serbia', 'Scotland', 'Slovakia', 'Ukraine',
    'Morocco', 'Senegal', 'Egypt', 'Nigeria', 'South Africa', 'DR Congo',
    'Cameroon', 'Mali', 'Algeria',
    'Japan', 'South Korea', 'Iran', 'Australia', 'Saudi Arabia',
    'Jordan', 'Iraq', 'Uzbekistan',
    'New Zealand', 'Kenya', 'Thailand',
]

# Get latest form snapshot for each WC 2026 team
latest_date = df['date'].max()
snapshot_rows = []

for team in WC2026_TEAMS:
    form = rolling_form(team, latest_date + pd.Timedelta(days=1), window=10)
    snapshot_rows.append({
        'team': team,
        'elo':  round(elo[team], 1),
        'wc_titles':    WC_TITLES.get(team, 0),
        'wc_apps':      WC_APPEARANCES.get(team, 0),
        'form_win_rate':       round(form['win_rate'], 3),
        'form_goals_scored':   round(form['goals_scored'], 2),
        'form_goals_conceded': round(form['goals_conceded'], 2),
    })

snapshot = pd.DataFrame(snapshot_rows).sort_values('elo', ascending=False).reset_index(drop=True)
print('Top 15 WC 2026 teams by ELO:')
print(snapshot.head(15).to_string(index=False))

Top 15 WC 2026 teams by ELO:
       team    elo  wc_titles  wc_apps  form_win_rate  form_goals_scored  form_goals_conceded
      Spain 1733.5          1       16            0.6                2.7                  0.5
  Argentina 1687.7          3       18            0.8                2.3                  0.3
     France 1669.0          2       16            0.8                2.4                  0.8
    England 1606.0          1       16            0.7                2.2                  0.5
     Brazil 1587.4          5       22            0.6                2.5                  1.1
    Germany 1582.0          4       20            0.9                2.8                  0.8
   Portugal 1576.8          0        9            0.6                2.6                  1.0
   Colombia 1565.7          0        7            0.7                2.5                  1.0
Netherlands 1558.2          0       11            0.6                2.8                  0.7
    Morocco 1548.4          0  

## 6. Save Outputs

In [9]:
features_df.to_csv(PROC / 'matches_features.csv', index=False)
snapshot.to_csv(PROC / 'elo_snapshot.csv', index=False)

sep = '=' * 52
print(sep)
print('  FEATURE ENGINEERING  —  SUMMARY')
print(sep)
print(f'  Training rows        : {len(features_df):>8,}')
print(f'  Features per match   : {len(FEATURE_COLS):>8}')
print(f'  Date range           : {str(features_df["date"].min())[:10]} to {str(features_df["date"].max())[:10]}')
print(f'  WC 2026 teams snapped: {len(snapshot):>8}')
print(sep)
print('  Saved:')
print('    data/processed/matches_features.csv')
print('    data/processed/elo_snapshot.csv')
print(sep)

  FEATURE ENGINEERING  —  SUMMARY
  Training rows        :   31,974
  Features per match   :       15
  Date range           : 1990-01-12 to 2026-06-07
  WC 2026 teams snapped:       48
  Saved:
    data/processed/matches_features.csv
    data/processed/elo_snapshot.csv
